# Notebook 6: Comprehensive LLM Evaluation

End-to-end LLM quality evaluation for the MedJuras RAG pipeline.

## Prerequisites
- **Notebook 1** — chunks indexed in Elasticsearch + Qdrant
- **Notebook 2** — `data/evaluation/ground_truth.json` (200 questions)
- Docker ES/Qdrant running (`local=True`)

## Evaluation tracks

### Track 1 — Offline hybrid RAG + LLM judge (`llm_judge.py`)
1. `hybrid_search` retrieves top-5 chunks (Qdrant + ES RRF)
2. Chat model generates an answer from context only
3. Judge scores **11 EU medico-legal criteria** (1–5 each)

### Track 2 — Agentic RAG evaluation (`llm_evaluation.py`)
1. Multi-iteration loop with `eu_hybrid_search` tool (same as production)
2. Judge scores **tool appropriateness**, **answer quality**, **information synthesis** (0–10)

## Models (env-dependent)

| Role | MyGenAssist (`MYGENASSIST_API_KEY`) | 
|------|-------------------------------------|
| Answer generation | `MYGENASSIST_CHAT_MODEL` → gpt-oss-120b |
| Judge | `MYGENASSIST_AUX_MODEL` → gpt-5.1 |

## Outputs (`results/`)
- `llm_evaluation.json` — combined summary
- `offline_rag_judge.json` — Track 1 (legacy name)
- `agentic_evaluation.json` — Track 2

## Cost control
Adjust `RAG_MAX_SAMPLES` and `AGENTIC_MAX_SAMPLES` below. Agentic eval is more expensive (multiple LLM + tool calls per question).

In [ ]:
import _bootstrap  # noqa: F401
import json

from evaluation.llm_evaluator import run_full_llm_evaluation, save_llm_evaluation_results

RAG_MAX_SAMPLES = 100
AGENTIC_MAX_SAMPLES = 40
LOCAL = True
COMPARE_APPROACHES = False  # True compares default/conservative/creative (costly)

results = run_full_llm_evaluation(
    rag_max_samples=RAG_MAX_SAMPLES,
    agentic_max_samples=AGENTIC_MAX_SAMPLES,
    local=LOCAL,
    compare_approaches=COMPARE_APPROACHES,
)
saved = save_llm_evaluation_results(results)

print("Summary:", json.dumps(results["summary"], indent=2))
print("Saved:", {k: str(v) for k, v in saved.items()})